In [2]:
import os
import time
import math

import cv2
import numpy as np


Dataset:
- https://spandh.dcs.shef.ac.uk//gridcorpus/

Medium Article:
- https://medium.com/towards-data-science/designing-your-neural-networks-a5e4617027ed


Inputs: 
- [ ] Organise Dataset of choice in files 
    - Silentframes - format: {filename}_sil_x.mpg
    - OneWordframes -format: {filename}_word_x.mpg x represents the line number in the file
    - Multiword frames?? - {stiched from one word frames using x value to verify continuity} 
- [ ] Generate consistent script to iterate through files
- [ ] Catogerise files into relevant categories (speaking frames: non-speaking frames)
- [ ] Remove excess pixels and standardise image size
- [ ] Numpyify and injest from model

Dataset of Choice is LipNet
Key folders:
S1 - contains video audio footage
Alignments/S1 - contains aligned audio transcriptions

In [3]:
LIPNET_S1_DIR = "/media/tymstr/Expansion/DiarizeDatabase/lipnetdata/s1"
LIPNET_ALIGNMENTS_DIR = "/media/tymstr/Expansion/DiarizeDatabase/lipnetdata/alignments/s1"

GRID_S1_DIR = "/media/tymstr/Expansion/DiarizeDatabase/gridcorpusdata/s1"
GRID_ALIGNMENTS_DIR = "/media/tymstr/Expansion/DiarizeDatabase/gridcorpusdata/align"

TEST_ALIGNMENT_DIR = "/media/tymstr/Expansion/DiarizeDatabase/testmodeldata/align"
TEST_S1_DIR = "/media/tymstr/Expansion/DiarizeDatabase/testmodeldata/S1"

VALIDATION_ALIGNMENT_DIR = "/media/tymstr/Expansion/DiarizeDatabase/validationdata/align"
VALIDATION_S1_DIR = "/media/tymstr/Expansion/DiarizeDatabase/validationdata/S2"

def getAlignFiles(ALIGNMENTS_DIR: str):
    files = []
    for file in os.listdir(ALIGNMENTS_DIR):
        files.append(file)
    return files

def getVideoFiles(S1_DIR: str):
    files = []
    for file in os.listdir(S1_DIR):
        files.append(file)

    return files

def deconstructAlign(dir, filename):
    alignment_array = []
    y = dir + "/" + filename
    with open(y, 'r') as file:
        for line in file:
            alignment_array.append(line.strip('\n').split(" "))
    
    return alignment_array

def deconstructS1(dir, i):
    x = getVideoFiles(dir)[i]
    y = dir + "/" + x
    video = cv2.VideoCapture(y)
    num_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    # print(num_frames)

    success = True
    count = 0
    while success:
        success, img = video.read()
        if success==False:
            break
        count += 1
        # cv2.imshow('image', img)
        # time.sleep(0.2)
        
        # cv2.waitKey(1)


    # cv2.waitKey(0)
    # cv2.destroyAllWindows()

    # print(count)
 
def calculateClips(filename, alignment_array):
    # alignment_array = deconstructAlign(GRID_ALIGNMENTS_DIR,0)
    # filename  = getAlignFiles(GRID_ALIGNMENTS_DIR)[0]
    clips = []
    for alignment in alignment_array:
        start, end, word = alignment
        start_frame = math.ceil(int(start)*0.001)
        end_frame = math.ceil(int(end)*0.001)
        # print("Start Frame:", start_frame, "End Frame:", end_frame, "Word:", word)

        clips.append([filename, start_frame, end_frame, word])

    return clips
# print(deconstructAlign(GRID_ALIGNMENTS_DIR,0))

# deconstructS1(GRID_S1_DIR,0)
def allClips(folder):
    all_clips = []#hold all clips in form [filename, start_frame, end_frame, word]
    #for all files in all folders
    for filename in getAlignFiles(folder):
        alignment_array = deconstructAlign(folder,filename)
        video_clips = calculateClips(filename, alignment_array)
        for clip in video_clips:
            all_clips.append(clip)

    return np.array(all_clips)

#Given a file, 0, 23000
#return frames?

# To determine the size of the input find the large amount of time a person is saying a single word (exclude silence?)
# Lipnet Organise Folder for classified data

Calculate Frame to alignment

Numofframes = cv2.CAP_PROP_FRAME_COUNT() // 
length of video (seconds) = ffmpeg -i bbaf2n.mpg 2>&1 | grep "Duration"


1.
Khz to Time
X khz / 25000 = time (seconds)
2.
Time to Frames
Time / Clip time = frame / num of frames
3.def:
khz to frames relationship
((X khz / 25000) / Clip time) x num of frames =~ frame

Given a .align for each line in .align
startframe, endframe = khztoframes(startvalue), khztoframes(endvalue)
if word == sil:
    add to non talking dataset
else:
    add to talking dataset

In [4]:
import tensorflow as tf
import mediapipe as mp

2025-02-24 14:16:34.172268: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1740406594.186401  951883 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1740406594.190844  951883 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-24 14:16:34.209739: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
def exampleFrame():
    dir = GRID_S1_DIR
    x = getVideoFiles(dir)[0]
    y = dir + "/" + x
    video = cv2.VideoCapture(y)
    video.set(cv2.CAP_PROP_POS_FRAMES, 0)
    res, frame = video.read()
    
    return frame
    

In [6]:
MOUTH_LANDMARKS = [0, 267, 269, 270, 409, 306, 375, 321, 405, 314, 17, 84, 181, 91, 146, 61, 185, 40, 39,
                                37]

mpDraw = mp.solutions.drawing_utils
mpFaceMesh = mp.solutions.face_mesh
faceMesh = mpFaceMesh.FaceMesh(max_num_faces=2)
drawSpec = mpDraw.DrawingSpec(thickness=1, circle_radius=1)

def findMouth(img, output):
    points = []
    dict = {}
    try:
        results = faceMesh.process(img)
        if results.multi_face_landmarks:
            for faceLms in results.multi_face_landmarks:
                # mpDraw.draw_landmarks(img, faceLms, mpFaceMesh.FACEMESH_CONTOURS, drawSpec, drawSpec)
                for id,lm in enumerate(faceLms.landmark):
                    image_height, image_width, image_channels = img.shape
                    x,y = int(lm.x*image_width), int(lm.y*image_height)
                    if id in MOUTH_LANDMARKS:
                        points.append((x,y))
                        dict[id] = (x,y)
    except:
        points.append((41,61))
    if output == "points":            
        return points
    else:
        return dict
    

I0000 00:00:1740406597.953864  951883 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1740406597.955937 3541136 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 24.0.9-0ubuntu0.1), renderer: NV194
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [7]:
# x = findMouth(exampleFrame(), "points")
# print(x)
# y, z = x[0]
# print(y, z)

W0000 00:00:1740406597.961156 3541123 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


W0000 00:00:1740406597.971823 3541122 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [ ]:
# Frames from video returns a numpy array

#large shufflable list
# list = [[filename, word, startframe, endframe], ...]  (This list will be defined/shuffled in Frame Generator)

#exampleframe = cv2.imread(image_path)

def formatFrames(frame, output_size):
    try:
        mouthpoints = findMouth(frame, "points")
        x,y = mouthpoints[0]
        # print(type(frame))
        #Resize image around the mouth
        new_frame = frame[y-40:y+60,x-50:x+50,:]
        
        # cv2.waitKey(0)
        # cv2.destroyAllWindows()
        #Convert into an acceptable format
    except:
        new_frame = frame[0:100, 0:100]


    new_frame = tf.image.convert_image_dtype(frame, tf.float32)
    new_frame = tf.image.resize_with_pad(frame, *output_size)
    return new_frame


# formatFrames(exampleFrame(), (100, 100))

In [9]:
#large shufflable list
# list = [[filename, word, startframe, endframe], ...]  (This list will be defined/shuffled in Frame Generator)
#Given a videopath
# calculateClips(video_path)
#AllClips(GRID_ALIGNMENTS_DIR)

import random
import numpy as np


def framesFromVideo(video_path, clip_name, clip_start, clip_end, clip_word):
    
    result = []
    full_path = video_path + "/" + clip_name
    
    # clip_end = int(clip_end)
    # clip_start - int(clip_start)
    video = cv2.VideoCapture(full_path)
    # print(full_path)

    # print(type(video))
    if int(clip_end) >= 11:
        for i in range(11, 1, -1):
            video.set(cv2.CAP_PROP_POS_FRAMES, int(clip_end)-i)
            res, frame = video.read()
            if not res:
                print("clip end:", clip_end)
                print("clip start:", clip_start)
                print("i:", i)

            result.append(formatFrames(frame, (100,100)))
    else:
        clip_length = min(int(clip_end) - int(clip_start), 10)
        
        # print("Clip length:" , clip_length)
        for i in range(clip_length):
            video.set(cv2.CAP_PROP_POS_FRAMES, int(clip_start)+i)
            res, frame = video.read()
            if not res:
                print("I've pissed me self in the else statement sir")
            result.append(formatFrames(frame, (100,100)))
        remaining_frames = [result[-1]] * (10-len(result))
        result += remaining_frames
        # print("Result length:" , len(result))
        # print("Remaining Frames length:" , len(remaining_frames))
        
        
        

    #return  a numpy array of the frames
    video.release()

    # if len(result) == 0:
    #     return np.array([])
    

    result = np.array(result)[..., [2,1,0]]

    return result


class FrameGenerator():
    def __init__(self, align_path,video_path, training=False):
        self.align_path = align_path
        self.video_path = video_path
        self.training = training
    pass


    def __call__(self):
        ALL_CLIPS = allClips(self.align_path)

    
        if self.training:
            np.random.shuffle(ALL_CLIPS)

        for clip_name, clips_start, clip_end, clip_word in ALL_CLIPS:
            video_clip_name = clip_name.split(".")[0]+".mpg"
            video_frames = framesFromVideo(self.video_path, video_clip_name, clips_start, clip_end, clip_word)
            if len(video_frames) == 0:
                # print("failed frame:" , video_clip_name)
                continue
            if clip_word != "sil":
                label = 1
            else:
                label = 0

            # print(video_clip_name, len(video_frames), label)
            yield video_frames, label

        #returns a set of frames with their associated label


In [10]:
fg = FrameGenerator(GRID_ALIGNMENTS_DIR, GRID_S1_DIR, True)

frames, label = next(fg())

print(f"Shape: {frames.shape}")
print(f"Label: {label}")

Shape: (10, 100, 100, 3)
Label: 0


W0000 00:00:1740406598.503287 3541127 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
2025-02-24 14:16:38.505024: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [11]:
output_signature = (tf.TensorSpec(shape = (None,None, None,3), dtype = tf.float32),
                    tf.TensorSpec(shape = (), dtype = tf.int16))
train_ds = tf.data.Dataset.from_generator(FrameGenerator(GRID_ALIGNMENTS_DIR, GRID_S1_DIR, True),
                                          output_signature = output_signature)
val_ds = tf.data.Dataset.from_generator(FrameGenerator(VALIDATION_ALIGNMENT_DIR, VALIDATION_S1_DIR, True),
                                          output_signature = output_signature)


In [12]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).cache().prefetch(buffer_size = AUTOTUNE)
val_ds = val_ds.shuffle(1000).cache().prefetch(buffer_size = AUTOTUNE)

In [13]:
train_ds = train_ds.batch(2)
val_ds = val_ds.batch(2)

In [14]:
# train_frames, train_labels = next(iter(train_ds),)
  
# val_frames, val_labels = next(iter(val_ds))clip_end

In [15]:
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Conv3D, LSTM, Dense, Dropout, Bidirectional, MaxPool3D, Activation, Reshape, SpatialDropout3D, BatchNormalization, TimeDistributed, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, LearningRateScheduler

### Model 2

In [16]:


# model2 = tf.keras.Sequential()
# model2.add(Conv3D(128, 3, input_shape=(3,100,100,3), padding='same'))
# model2.add(Activation('relu'))
# model2.add(MaxPool3D((1,2,2)))clip_length

# model2.add(Conv3D(256, 3, padding='same'))
# model2.add(Activation('relu'))
# model2.add(MaxPool3D((1,2,2)))

# # model2.add(Conv3D(75, 3, padding='same'))
# # model2.add(Activation('relu'))
# # model2.add(MaxPool3D((1,2,2)))

# model2.add(TimeDistributed(Flatten()))

# model2.add(Bidirectional(LSTM(128, kernel_initializer='Orthogonal', return_sequences=True)))
# model2.add(Dropout(.5))

# model2.add(Bidirectional(LSTM(128, kernel_initializer='Orthogonal', return_sequences=True)))
# model2.add(Dropout(.5))

# model2.add(Dense(1, kernel_initializer='he_normal', activation='sigmoid'))

In [17]:

# model2.compile(optimizer = 'adam',
#               loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits = True),
#               metrics=['accuracy'])

# model2.fit(train_ds, 
#           epochs = 10,
#           validation_data = val_ds,
#           callbacks = tf.keras.callbacks.EarlyStopping(patience = 2, monitor = 'val_loss'))

### Model 1

In [18]:
# net = tf.keras.applications.EfficientNetB0(include_top = False)
# net.trainable = False

# model = tf.keras.Sequential([
#     tf.keras.layers.Rescaling(scale=255),
#     tf.keras.layers.TimeDistributed(net),
#     tf.keras.layers.Dense(10),
#     tf.keras.layers.GlobalAveragePooling3D()
# ])

# model.compile(optimizer = 'adam',
#               loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits = True),
#               metrics=['accuracy'])

# model.fit(train_ds, 
#           epochs = 10,
#           validation_data = val_ds,
#           callbacks = tf.keras.callbacks.EarlyStopping(patience = 2, monitor = 'val_loss'))


### Model 3

In [19]:
from tensorflow.keras import layers, models

# model3 = models.Sequential([
#     layers.ConvLSTM2D(filters=16, kernel_size=(3, 3), padding='same', return_sequences=True, input_shape=(10, 100, 100, 3)),
#     layers.BatchNormalization(),
#     layers.ConvLSTM2D(filters=32, kernel_size=(3, 3), padding='same', return_sequences=True),
#     layers.BatchNormalization(),
#     layers.ConvLSTM2D(filters=32, kernel_size=(3, 3), padding='same', return_sequences=False),
#     layers.BatchNormalization(),
#     layers.Flatten(),
#     # layers.Dense(128, activation='relu'),
#     layers.Dense(64, activation='relu'),
#     layers.Dense(1, activation='sigmoid')
# ])

# model3.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# model3.fit(train_ds, 
#           epochs = 10,
#           validation_data = val_ds,
#           callbacks = tf.keras.callbacks.EarlyStopping(patience = 2, monitor = 'val_loss'))


### Model 4

In [20]:
input_shape = (10, 100, 100, 3)

model4 = models.Sequential([
    # Input Layer
    layers.Input(shape=input_shape),

    # (2+1)D Conv Block 1
    layers.Conv3D(32, kernel_size=(1, 3, 3), padding='same', activation='relu'),
    layers.Conv3D(32, kernel_size=(3, 1, 1), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling3D(pool_size=(1, 2, 2)),

    # (2+1)D Conv Block 2
    layers.Conv3D(64, kernel_size=(1, 3, 3), padding='same', activation='relu'),
    layers.Conv3D(64, kernel_size=(3, 1, 1), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling3D(pool_size=(2, 2, 2)),

    # (2+1)D Conv Block 3
    layers.Conv3D(128, kernel_size=(1, 3, 3), padding='same', activation='relu'),
    layers.Conv3D(128, kernel_size=(3, 1, 1), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling3D(pool_size=(2, 2, 2)),

    # Global Feature Aggregation
    layers.GlobalAveragePooling3D(),
    layers.Dropout(0.5),

    # Fully Connected Layer
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    # Output Layer (Binary Classification)
    layers.Dense(1, activation='sigmoid')  # Change from softmax to sigmoid
])

# Compile Model
model4.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(),  # Change loss function
              metrics=['accuracy'])

# Model Summary
model4.fit(train_ds, 
          epochs = 2,
          validation_data = val_ds,
          callbacks = tf.keras.callbacks.EarlyStopping(patience = 2, monitor = 'val_loss'))

Epoch 1/2


2025-02-24 14:16:51.631950: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:4: Filling up shuffle buffer (this may take a while): 84 of 1000
2025-02-24 14:17:11.565989: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:4: Filling up shuffle buffer (this may take a while): 240 of 1000
2025-02-24 14:17:21.594869: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:4: Filling up shuffle buffer (this may take a while): 323 of 1000
2025-02-24 14:17:41.628925: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:4: Filling up shuffle buffer (this may take a while): 489 of 1000
2025-02-24 14:18:01.665908: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:4: Filling up shuffle buffer (this may take a while): 658 of 1000
2025-02-24 14:18:21.585128: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:4: Filling up shuffle buffer (this may take a while):

   1694/Unknown 1082s 566ms/step - accuracy: 0.7276 - loss: 0.6115

[mpeg1video @ 0x7838ec0d7600] ac-tex damaged at 22 17
[mpeg1video @ 0x7838ec0d7600] Warning MVs not available
[mpeg1video @ 0x7838ec0d7600] ac-tex damaged at 22 17
[mpeg1video @ 0x7838ec0d7600] Warning MVs not available
[mpeg1video @ 0x7838ec0d7600] ac-tex damaged at 22 17
[mpeg1video @ 0x7838ec0d7600] Warning MVs not available


   4012/Unknown 2396s 567ms/step - accuracy: 0.7402 - loss: 0.5940

2025-02-24 14:56:34.880385: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
/home/tymstr/.local/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:151: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()
2025-02-24 14:56:45.162413: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:7: Filling up shuffle buffer (this may take a while): 66 of 1000
2025-02-24 14:57:05.162977: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:7: Filling up shuffle buffer (this may take a while): 216 of 1000
2025-02-24 14:57:15.168860: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:7: Filling up shuff

4012/4012 ━━━━━━━━━━━━━━━━━━━━ 3436s 826ms/step - accuracy: 0.7402 - loss: 0.5940 - val_accuracy: 0.7507 - val_loss: 0.5653
Epoch 2/2


2025-02-24 15:13:55.023110: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]


4012/4012 ━━━━━━━━━━━━━━━━━━━━ 2409s 600ms/step - accuracy: 0.7506 - loss: 0.5687 - val_accuracy: 0.7507 - val_loss: 0.5620


2025-02-24 15:54:03.979296: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]


In [43]:
test_samples = []
prediction = model4.predict(next(iter(train_ds)))
print(prediction[0][0])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
[[0.72451866]
 [0.72522163]]


How does a 3D convNN injest video?
https://www.tensorflow.org/tutorials/load_data/video

then

https://www.tensorflow.org/tutorials/video/video_classification

then consider

MoviNet for streaming action recognition


Outputs:
One neuron with the probalility of the positive class

Layers:
- Lookup YOLO ResNet and VGG for processing image and speech data
-Same number of neurons for all layers should suffice, to increase accuracy consider using a large first layer filtered into smaller layers.(later) More layer > more neurons
- I’d recommend starting with 1–5 layers and 1–100 neurons and slowly adding more layers and neurons until you start overfitting

Assessing Layers:
- You can track your loss and accuracy within your Weights and Biases dashboard to see which hidden layers + hidden neurons combo leads to the best loss
- Andrej Karpathy also recommends the overfit then regularize approach — “first get a model large enough that it can overfit (i.e. focus on training loss) and then regularize it appropriately (give up some training loss to improve the validation loss).”
-

Loss Function:
- Cross-entropy will serve you well in most cases.

Batch Size:
- Smaller may often be better

Number of epochs:
- Start with an early number of epochs and use **Early Stopping** to halt training when performance stops improving

Scaling features:
- currently scaling based on distance from point on mouth using pixels, consider a scaling relative to the mouth landmarks??


Learning Rate:
- To find the best learning rate, start with a very low value (10^-6) and slowly multiply it by a constant until it reaches a very high value (e.g. 10). Measure your model performance (vs the log of your learning rate) in your Weights and Biases dashboard to determine which rate served you well for your problem.
- The best learning rate is usually half of the learning rate that causes the model to diverge. Feel free to set different values for learn_rate in the accompanying code and seeing how it affects model performance to develop your intuition around learning rates.
- recommend using the Learning Rate finder method proposed by Leslie Smith.

Momentum:
- Gradient descent to consider momentum and local minimas
- In general, you want your momentum value to be very close to one. 0.9 is a good place to start for smaller datasets

Vanishing + Exploding Gradients:
- Activation Functions
    - User ReLU for now
    - Revisit if unhappy with networks learning rate
